# DCAE — Image Compression on ImageNet (MMS 2025/26, Topic 3)

**Run end-to-end with: Runtime → Run all**

## Before you run (one-time setup)

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **HuggingFace token** (needed to stream ImageNet-1k):
   - Accept the dataset licence at `huggingface.co/datasets/ILSVRC/imagenet-1k`
   - Create a **Read** token at `huggingface.co/settings/tokens`
   - Left sidebar → 🔑 **Secrets** → Add new secret: Name = `HF_TOKEN`, Value = your token → toggle **Notebook access ON**
3. **Google Drive**: the notebook mounts it automatically — just click *Allow* when prompted. Checkpoints and results will be saved there so a disconnect doesn't lose work.

Everything else is automated.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  CELL 0 — PARAMETERS  (edit here if needed)     ║
# ╚══════════════════════════════════════════════════╝

GITHUB_REPO = "https://github.com/hhrnjic1/DCAE.git"
GIT_BRANCH  = "imagenet-port"

# ImageNet subset size (HF streaming)
PER_CLASS    = 4     # images per class in train split (~4 000 total, 1 000 classes)
IN_TEST_SIZE = 100   # images held out for the ImageNet eval

# Fine-tuning knobs
FT_LAMBDA = "0.0067"   # lambda point to fine-tune (mid-rate MSE checkpoint)
FT_EPOCHS = 1          # 1 epoch over the small subset is enough to show training ran

# Where to persist checkpoints + results on Drive
DRIVE_DIR = "/content/drive/MyDrive/DCAE_run"

# Pretrained MSE checkpoint Google Drive IDs (from the paper's README)
CKPT_IDS = {
    "0.0018": "1JzVuERiZe8cStgLnE5TJii_ssppgY1p-",
    "0.0035": "1JE0SO876a-btXzOQLTilj7D0vJdePlB4",
    "0.0067": "1LdycatKcGXHvFjoR-NE-GWnPlL9-BRWX",
    "0.013":  "1kXfvxsljdN3EfXDGqzknFc2Ecsgf8qgS",
    "0.025":  "1-6ZZ-bScGYj448h1sqMTX4w2Q75MQQ1q",
    "0.05":   "1jCsRJq7Ttx22-yWQbEQAHJWbdIDtc30k",
}

CKPT_DIR   = "/content/ckpts"
REPO_DIR   = "/content/DCAE"
IMAGENET_DIR = "/content/imagenet_dcae"

print("Parameters set.")

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 1 — GPU CHECK             ║
# ╚══════════════════════════════════╝
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU and re-run.")

gpu_name = result.stdout.strip()
print(f"GPU: {gpu_name}")
if "T4" not in gpu_name and "A100" not in gpu_name and "V100" not in gpu_name:
    print("WARNING: not a T4 — you may hit memory limits. T4 is recommended.")
else:
    print("Good to go.")

In [ ]:
# ╔══════════════════════════════════╗
# ║  CELL 2 — MOUNT GOOGLE DRIVE    ║
# ╚══════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/finetune", exist_ok=True)
print(f"Drive mounted. Results will be mirrored to: {DRIVE_DIR}")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 3 — CLONE FORK + INSTALL DEPENDENCIES         ║
# ╚══════════════════════════════════════════════════════╝
import os

if not os.path.exists(REPO_DIR):
    !git clone -b {GIT_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — skipping clone.")

%cd {REPO_DIR}

!pip install -q \
    "compressai==1.2.6" \
    timm \
    einops \
    pytorch-msssim \
    thop \
    tensorboard \
    datasets \
    huggingface_hub \
    gdown \
    numpy \
    matplotlib

print("\nInstall complete.")

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 4 — SMOKE TEST (model forward pass)   ║
# ╚══════════════════════════════════════════════╝
# Confirms DCAE loads and runs on the GPU before we spend time on data/training.
!python scripts/smoke_test.py

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — HF LOGIN + BUILD ImageNet SUBSET (streaming)      ║
# ╚══════════════════════════════════════════════════════════════╝
# Requires HF_TOKEN secret set in the left sidebar (see setup notes).
# Streams imagenet-1k directly from HuggingFace — no full download needed.

import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise ValueError(
        "HF_TOKEN not found! Add it via left sidebar → Secrets → Name=HF_TOKEN, "
        "Value=your_token, Notebook access ON.  "
        "Also accept the imagenet-1k licence at huggingface.co/datasets/ILSVRC/imagenet-1k"
    )
login(token=hf_token, add_to_git_credential=False)
print("Logged in to HuggingFace.")

train_done = os.path.isdir(f"{IMAGENET_DIR}/train") and len(os.listdir(f"{IMAGENET_DIR}/train")) > 0
test_done  = os.path.isdir(f"{IMAGENET_DIR}/test")  and len(os.listdir(f"{IMAGENET_DIR}/test"))  > 0

if train_done and test_done:
    print(f"ImageNet subset already built at {IMAGENET_DIR} — skipping.")
    n_train = len(os.listdir(f"{IMAGENET_DIR}/train"))
    n_test  = len(os.listdir(f"{IMAGENET_DIR}/test"))
    print(f"  train: {n_train} images,  test: {n_test} images")
else:
    print(f"Streaming ImageNet-1k ({PER_CLASS} imgs/class × 1000 classes + {IN_TEST_SIZE} test imgs)...")
    print("This is the longest step — ~20-40 min depending on HF bandwidth. Grab a coffee.")
    !python scripts/build_imagenet_subset.py \
        --source hf \
        --out_dir {IMAGENET_DIR} \
        --per_class {PER_CLASS} \
        --test_size {IN_TEST_SIZE} \
        --min_side 256 \
        --max_side 512

print("ImageNet subset ready.")

In [ ]:
# ╔════════════════════════════════════════════════════╗
# ║  CELL 6 — DOWNLOAD 6 PRETRAINED MSE CHECKPOINTS   ║
# ╚════════════════════════════════════════════════════╝
import os

os.makedirs(CKPT_DIR, exist_ok=True)

for lmbda, drive_id in CKPT_IDS.items():
    dst = f"{CKPT_DIR}/{lmbda}.pth.tar"
    if os.path.exists(dst):
        print(f"  lambda={lmbda}: already downloaded — skipping.")
    else:
        print(f"  lambda={lmbda}: downloading from Google Drive...")
        !gdown --id {drive_id} -O {dst} --quiet
        if os.path.exists(dst):
            size_mb = os.path.getsize(dst) / 1e6
            print(f"    -> {dst}  ({size_mb:.1f} MB)")
        else:
            print(f"    WARNING: download may have failed for lambda={lmbda}")

print("\nAll checkpoints ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — BUILD PAPER ANCHOR FILES for BD-Rate                  ║
# ║  RD_data.json uses dict-of-arrays; bd_rate.py needs list-of-dicts║
# ╚══════════════════════════════════════════════════════════════════╝
import json, os

os.makedirs("reports", exist_ok=True)

with open("RD_data.json") as f:
    rd = json.load(f)

# Kodak PSNR anchor
kodak = rd["Kodak"]
paper_kodak = [{"bpp": b, "psnr": p}
               for b, p in zip(kodak["bpp"], kodak["PSNR"])]
with open("reports/paper_kodak.json", "w") as f:
    json.dump(paper_kodak, f, indent=2)
print("reports/paper_kodak.json written:", paper_kodak)

# Kodak MS-SSIM anchor
kodak_ms = rd["Kodak_MS-SSIM"]
paper_kodak_msssim = [{"bpp": b, "msssim": m}
                      for b, m in zip(kodak_ms["bpp"], kodak_ms["MS_SSIM"])]
with open("reports/paper_kodak_msssim.json", "w") as f:
    json.dump(paper_kodak_msssim, f, indent=2)
print("reports/paper_kodak_msssim.json written.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — EVALUATE ALL 6 CHECKPOINTS ON KODAK               ║
# ║  Uses real entropy coding (--real) so bpp is actual file size║
# ╚══════════════════════════════════════════════════════════════╝
import os

os.makedirs("reports", exist_ok=True)

for lmbda in CKPT_IDS:
    ckpt = f"{CKPT_DIR}/{lmbda}.pth.tar"
    out  = f"reports/kodak_{lmbda}.json"
    if not os.path.exists(ckpt):
        print(f"SKIP lambda={lmbda}: checkpoint not found at {ckpt}")
        continue
    print(f"\n{'='*60}")
    print(f"Evaluating lambda={lmbda} on Kodak (real entropy coding)...")
    print(f"{'='*60}")
    !python eval.py \
        --checkpoint {ckpt} \
        --data datasets/kodak \
        --cuda \
        --real \
        --lmbda {lmbda} \
        --results_json {out}

print("\nKodak evaluation complete. JSON files in reports/")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — EVALUATE ALL 6 CHECKPOINTS ON ImageNet TEST SPLIT     ║
# ║  This is the "results on ImageNet" the assignment asks for.      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

imagenet_test = f"{IMAGENET_DIR}/test"
if not os.path.isdir(imagenet_test) or len(os.listdir(imagenet_test)) == 0:
    raise RuntimeError("ImageNet test split not found — did Cell 5 complete successfully?")

for lmbda in CKPT_IDS:
    ckpt = f"{CKPT_DIR}/{lmbda}.pth.tar"
    out  = f"reports/imagenet_{lmbda}.json"
    if not os.path.exists(ckpt):
        print(f"SKIP lambda={lmbda}: checkpoint not found.")
        continue
    print(f"\n{'='*60}")
    print(f"Evaluating lambda={lmbda} on ImageNet test split (real entropy coding)...")
    print(f"{'='*60}")
    !python eval.py \
        --checkpoint {ckpt} \
        --data {imagenet_test} \
        --cuda \
        --real \
        --lmbda {lmbda} \
        --results_json {out}

print("\nImageNet evaluation complete.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — BD-RATE vs the paper's reference curve           ║
# ║  Negative % = our eval matches or beats the paper.          ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json

# Collect available Kodak result files
kodak_jsons = [f"reports/kodak_{l}.json" for l in CKPT_IDS if os.path.exists(f"reports/kodak_{l}.json")]
if len(kodak_jsons) < 4:
    print(f"WARNING: only {len(kodak_jsons)} Kodak result files found — BD-Rate needs >=4 points for a reliable polynomial fit.")
elif len(kodak_jsons) < 2:
    print("Too few result files to compute BD-Rate. Skipping.")
else:
    ours_arg = " ".join(kodak_jsons)
    print("--- BD-Rate (PSNR) vs paper ---")
    !python scripts/bd_rate.py \
        --ours {ours_arg} \
        --ref_json reports/paper_kodak.json \
        --metric psnr \
        --out reports/bdrate_kodak_psnr.csv

    print("\n--- BD-Rate (MS-SSIM) vs paper ---")
    # MS-SSIM anchor only has 6 points from the MS-SSIM-trained models;
    # our MSE-trained models still report msssim so we can do a rough comparison.
    !python scripts/bd_rate.py \
        --ours {ours_arg} \
        --ref_json reports/paper_kodak_msssim.json \
        --metric msssim \
        --out reports/bdrate_kodak_msssim.csv

print("\nBD-Rate computation complete. CSVs in reports/")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — SHORT FINE-TUNE ON ImageNet SUBSET               ║
# ║  One epoch from the pretrained FT_LAMBDA checkpoint.        ║
# ║  Checkpoints saved to Drive every 200 iters — disconnect-safe║
# ╚══════════════════════════════════════════════════════════════╝
import os

ft_ckpt    = f"{CKPT_DIR}/{FT_LAMBDA}.pth.tar"
ft_savedir = f"{DRIVE_DIR}/finetune/"
imagenet_train = f"{IMAGENET_DIR}"

if not os.path.exists(ft_ckpt):
    raise FileNotFoundError(f"Pretrained checkpoint not found: {ft_ckpt}  (did Cell 6 run?)")

if not os.path.isdir(f"{IMAGENET_DIR}/train") or len(os.listdir(f"{IMAGENET_DIR}/train")) == 0:
    raise RuntimeError("ImageNet train split missing — did Cell 5 complete?")

# compressai.datasets.ImageFolder expects the root to contain train/ and test/
print(f"Fine-tuning lambda={FT_LAMBDA} for {FT_EPOCHS} epoch(s) on ImageNet subset (~{PER_CLASS*1000} images)")
print(f"Checkpoints → {ft_savedir}")
print("(If Colab disconnects, re-run this cell — it will resume from the last interval checkpoint.)\n")

# Check if a resumable interval checkpoint already exists
interval_ckpt = f"{ft_savedir}{FT_LAMBDA}/checkpoint_latest.pth.tar"
resume_ckpt = interval_ckpt if os.path.exists(interval_ckpt) else ft_ckpt
print(f"Starting from: {resume_ckpt}")

!python train.py \
    -d {imagenet_train} \
    --cuda \
    --amp \
    -e {FT_EPOCHS} \
    --lr_epoch {FT_EPOCHS} \
    --lambda {FT_LAMBDA} \
    --batch-size 8 \
    --num-workers 4 \
    --patch-size 256 256 \
    --save_interval 200 \
    --checkpoint {resume_ckpt} \
    --save_path {ft_savedir}

print("\nFine-tuning complete.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — RE-EVALUATE FINE-TUNED CHECKPOINT                ║
# ║  Before/after comparison on Kodak + ImageNet test.          ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json

ft_best = f"{DRIVE_DIR}/finetune/{FT_LAMBDA}/checkpoint_best.pth.tar"
ft_latest = f"{DRIVE_DIR}/finetune/{FT_LAMBDA}/checkpoint_latest.pth.tar"

# Use best checkpoint if available, otherwise latest
if os.path.exists(ft_best):
    ft_eval_ckpt = ft_best
    print(f"Using best fine-tuned checkpoint: {ft_best}")
elif os.path.exists(ft_latest):
    ft_eval_ckpt = ft_latest
    print(f"Using latest fine-tuned checkpoint: {ft_latest}")
else:
    raise FileNotFoundError("No fine-tuned checkpoint found in Drive — did Cell 11 complete?")

# Eval fine-tuned on Kodak
print("\n--- Fine-tuned: Kodak ---")
!python eval.py \
    --checkpoint {ft_eval_ckpt} \
    --data datasets/kodak \
    --cuda --real \
    --lmbda {FT_LAMBDA} \
    --results_json reports/kodak_{FT_LAMBDA}_ft.json

# Eval fine-tuned on ImageNet test
print("\n--- Fine-tuned: ImageNet test split ---")
!python eval.py \
    --checkpoint {ft_eval_ckpt} \
    --data {IMAGENET_DIR}/test \
    --cuda --real \
    --lmbda {FT_LAMBDA} \
    --results_json reports/imagenet_{FT_LAMBDA}_ft.json

# ── Print before/after table ──
def load_metrics(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

before_kodak    = load_metrics(f"reports/kodak_{FT_LAMBDA}.json")
after_kodak     = load_metrics(f"reports/kodak_{FT_LAMBDA}_ft.json")
before_inet     = load_metrics(f"reports/imagenet_{FT_LAMBDA}.json")
after_inet      = load_metrics(f"reports/imagenet_{FT_LAMBDA}_ft.json")

print(f"\n{'='*65}")
print(f"  Fine-tune comparison  (lambda={FT_LAMBDA})")
print(f"{'='*65}")
print(f"  {'Dataset':<18} {'Stage':<12} {'bpp':>7} {'PSNR(dB)':>10} {'MS-SSIM':>10}")
print(f"  {'-'*63}")

for tag, before, after in [("Kodak", before_kodak, after_kodak), ("ImageNet-test", before_inet, after_inet)]:
    for stage, m in [("pretrained", before), ("fine-tuned", after)]:
        if m:
            print(f"  {tag:<18} {stage:<12} {m['bpp']:>7.4f} {m['psnr']:>10.2f} {m['msssim']:>10.4f}")
        else:
            print(f"  {tag:<18} {stage:<12} {'N/A':>7}")

print(f"{'='*65}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — RD CURVE PLOT + SUMMARY TABLE + SAVE TO DRIVE    ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os, shutil
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 120})

# ── Load paper reference (Kodak) ──
with open("reports/paper_kodak.json") as f:
    paper = json.load(f)
paper_bpp  = [p["bpp"]  for p in paper]
paper_psnr = [p["psnr"] for p in paper]

# ── Load our Kodak eval points ──
our_bpp, our_psnr, our_msssim = [], [], []
for lmbda in sorted(CKPT_IDS.keys(), key=float):
    path = f"reports/kodak_{lmbda}.json"
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        our_bpp.append(d["bpp"])
        our_psnr.append(d["psnr"])
        our_msssim.append(d["msssim"])

# ── Load our ImageNet eval points ──
inet_bpp, inet_psnr = [], []
for lmbda in sorted(CKPT_IDS.keys(), key=float):
    path = f"reports/imagenet_{lmbda}.json"
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        inet_bpp.append(d["bpp"])
        inet_psnr.append(d["psnr"])

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Kodak PSNR
ax = axes[0]
ax.plot(paper_bpp, paper_psnr, "k--o", label="DCAE (paper)", linewidth=1.5)
if our_bpp:
    ax.plot(our_bpp, our_psnr, "b-s", label="DCAE (our eval)", linewidth=1.5)
ax.set_xlabel("Bit-rate (bpp)")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Rate-Distortion: Kodak")
ax.legend()
ax.grid(True, alpha=0.3)

# Right: ImageNet PSNR
ax = axes[1]
if inet_bpp:
    ax.plot(inet_bpp, inet_psnr, "g-^", label="DCAE (ImageNet test)", linewidth=1.5)
if our_bpp:
    ax.plot(our_bpp, our_psnr, "b--s", label="DCAE (Kodak)", linewidth=1.5, alpha=0.6)
ax.set_xlabel("Bit-rate (bpp)")
ax.set_ylabel("PSNR (dB)")
ax.set_title("Rate-Distortion: ImageNet-1k (test subset)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reports/rd_curve.png", bbox_inches="tight")
plt.show()
print("Plot saved to reports/rd_curve.png")

# ── Summary table ──
lambdas_sorted = sorted(CKPT_IDS.keys(), key=float)
print(f"\n{'='*75}")
print(f"  {'lambda':<8} {'Kodak bpp':>10} {'Kodak PSNR':>12} {'Kodak MS-SSIM':>14} {'ImageNet bpp':>13} {'ImageNet PSNR':>14}")
print(f"  {'-'*73}")

for i, lmbda in enumerate(lambdas_sorted):
    k_b = f"{our_bpp[i]:.4f}"    if i < len(our_bpp)   else "N/A"
    k_p = f"{our_psnr[i]:.2f}"   if i < len(our_psnr)  else "N/A"
    k_m = f"{our_msssim[i]:.4f}" if i < len(our_msssim) else "N/A"
    i_b = f"{inet_bpp[i]:.4f}"   if i < len(inet_bpp)  else "N/A"
    i_p = f"{inet_psnr[i]:.2f}"  if i < len(inet_psnr) else "N/A"
    print(f"  {lmbda:<8} {k_b:>10} {k_p:>12} {k_m:>14} {i_b:>13} {i_p:>14}")

print(f"{'='*75}")

# ── Mirror all reports to Drive ──
print(f"\nCopying reports to {DRIVE_DIR}/reports/ ...")
drive_reports = f"{DRIVE_DIR}/reports"
os.makedirs(drive_reports, exist_ok=True)
for fname in os.listdir("reports"):
    src = f"reports/{fname}"
    dst = f"{drive_reports}/{fname}"
    shutil.copy2(src, dst)
    print(f"  {fname}")

print(f"\nAll done! Results mirrored to {drive_reports}")
print("Use reports/rd_curve.png and reports/*.json in your IEEE write-up.")
print("BD-Rate CSV is at reports/bdrate_kodak_psnr.csv")